# Week 3 Practical Notebook  
## Tables, Variables, and Exploratory Data Analysis (EDA)

**Course:** WQD7013 - Statistics for Data Science  
**Program Level:** Master's  
**Prepared by:** Dr. Hamza Altarturi  
**Purpose:** Practical notebook for Week 3, designed to support hands-on learning in tabular data structure, variable types, univariate summaries, visualisation, and introductory exploratory data analysis using Python.

---

### Overview
This notebook is intended as a guided practical resource for students in **WQD7013 - Statistics for Data Science**.  
It introduces the foundational workflow of exploratory data analysis (EDA), with emphasis on:

- understanding rows, columns, cases, and variables
- distinguishing categorical and numerical variables
- checking data quality and missing values
- summarising one variable at a time
- choosing appropriate visualisations
- comparing groups and communicating findings clearly

The notebook is structured for students from mixed academic backgrounds and is designed to be:
- **practical**
- **interpretive**
- **reproducible**
- **GitHub-friendly**

> **Recommended use:** Run the notebook sequentially, read the interpretation prompts carefully, and complete the exercises at the end before moving to the next topic.


## Datasets used in this notebook

To keep this notebook fully runnable offline, we use built-in datasets from `plotly.express`:

- **Iris** → clean tabular data for variable types, summaries, histograms, box plots, and scatter plots
- **Tips** → everyday example for grouped comparisons and derived variables
- **Gapminder** → global data with strong skewness, scale issues, and richer interpretation

## Core vs enrichment
Throughout this notebook, sections are labeled as:

- **Core** → everyone should understand these
- **Recommended** → useful extensions
- **Enrichment** → optional deeper insight

This makes the notebook easier to use for students with different coding backgrounds.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

plt.rcParams["figure.figsize"] = (8, 5)
sns.set_theme(style="whitegrid")

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

## 1. Load the datasets

In [ ]:
iris = px.data.iris()
tips = px.data.tips()
gapminder = px.data.gapminder()

print("iris shape:", iris.shape)
print("tips shape:", tips.shape)
print("gapminder shape:", gapminder.shape)

In [ ]:
iris.head()

### Quick reflection
Before moving on, answer mentally:

- In `iris`, what does **one row** represent?
- Which columns look numerical?
- Which columns look categorical?
- Which column looks like an identifier rather than a measurement?

## 2. First 60 seconds with a new dataset (**Core**)

When you open a new dataset, do not jump straight to plotting or modeling. Start with a short inspection routine:

- `shape` → how many rows and columns?
- `head()` → what do the first rows look like?
- `info()` → what are the data types and missing-value counts?
- `describe()` → what do quick summaries look like?

This is often enough to catch obvious issues early.

In [ ]:
def inspect_dataset(df, name="dataset"):
    print(f"--- {name} ---")
    print("shape:", df.shape)
    print("\ncolumns:")
    print(df.columns.tolist())
    print("\ninfo:")
    print(df.info())
    print("\nsummary:")
    display(df.describe(include="all").T)

inspect_dataset(iris, "iris")

### Interpretation prompt
Write 2–3 sentences:

1. What does one row in `iris` represent?
2. Which variables are numerical?
3. Which variables are categorical or identifier-like?

## 3. Cases, variables, rows, and columns (**Core**)

A dataset is easier to understand when you separate these ideas:

- **Case / observation**: one unit described by the data
- **Variable**: one characteristic recorded for each case
- **Row**: one case
- **Column**: one variable

For `iris`:
- one row = one flower
- columns such as `sepal_length` and `petal_width` = measurements
- `species` = category
- `species_id` = code / identifier-like variable

> A very common mistake is to average an ID code as if it were a meaningful numerical variable.

## 4. Variable types (**Core**)

The first analytical decision is usually: **what kind of variable is this?**

### Main types
- **Categorical**
  - labels, groups, or classes
  - examples: species, smoker, day, continent
- **Numerical**
  - meaningful arithmetic
  - examples: bill amount, tip amount, sepal length

### More detail
- **Discrete numerical**: counts (e.g., party size)
- **Continuous numerical**: measurements (e.g., length, weight, GDP per capita)
- **Ordinal categorical**: ordered categories (e.g., low / medium / high)
- **Identifier**: labels like IDs that may look numeric but should not usually be averaged

In [ ]:
variable_guide = pd.DataFrame({
    "column": iris.columns,
    "suggested_type": [
        "numerical (continuous)",
        "numerical (continuous)",
        "numerical (continuous)",
        "numerical (continuous)",
        "categorical",
        "identifier-like"
    ]
})
variable_guide

### Mini-checkpoint
Classify these from the `tips` dataset before running code:

- `total_bill`
- `tip`
- `sex`
- `smoker`
- `day`
- `time`
- `size`

Then run the next cells and compare your answers.

In [ ]:
tips.head()

In [ ]:
tips.dtypes

## 5. Match variable type to the right summary and plot (**Core**)

This is one of the most useful practical rules in EDA.

| Variable situation | Good summaries | Good plots |
|---|---|---|
| One categorical | counts, proportions | bar chart |
| One numerical | mean/median, SD/IQR, quantiles | histogram, box plot |
| Two categorical | contingency table, row/column proportions | grouped or stacked bars |
| Two numerical | correlation direction/form/strength | scatter plot |
| Numerical by categorical | compare centers and spread across groups | side-by-side box plots |

> Before choosing a chart, ask: **What kinds of variables am I working with?**

## 6. Data-quality checks beyond missingness (**Core**)

EDA is not just making graphs. It is also checking whether the data look trustworthy.

Useful checks:
- missing values
- duplicated rows
- impossible or suspicious values
- inconsistent labels (`Male`, `male`, `M`)
- wrong data types
- strange ranges

Remember:
- **missing** is not the same as **zero**
- **outlier** is not automatically **error**
- **error** is not automatically **missing**

In [ ]:
def quality_report(df):
    report = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing_count": df.isna().sum(),
        "missing_pct": df.isna().mean().mul(100).round(2),
        "n_unique": df.nunique(dropna=True)
    })
    return report.sort_values(["missing_count", "n_unique"], ascending=[False, True])

quality_report(tips)

In [ ]:
print("Duplicated rows in tips:", tips.duplicated().sum())
print("Duplicated rows in iris:", iris.duplicated().sum())

### Try it yourself
Look at the quality report for `tips`.

- Which variables are categorical?
- Are there missing values?
- Does any column look suspicious immediately?
- Why might `size` be treated as discrete numerical?

## 7. Missing data in practice (**Core**)

The built-in datasets here are mostly clean. Real datasets often are not.

To demonstrate missing-data handling, we create a copy of `tips` and deliberately inject a few missing values.

In [ ]:
tips_missing = tips.copy()

tips_missing.loc[[3, 7, 12], "tip"] = np.nan
tips_missing.loc[[5, 11], "sex"] = np.nan
tips_missing.loc[[9], "total_bill"] = np.nan

quality_report(tips_missing)

In [ ]:
tips_missing.head(15)

### Why this matters
Missing values can silently change results:

- summaries may drop rows without you noticing
- plots may use fewer observations than the full dataset
- comparisons across groups may become unfair if missingness is systematic

### Good habit
Whenever you summarize or visualize, ask:
- How many observations were actually used?
- Are missing values random, or might they bias the result?

## 8. Quick selection, filtering, and sorting (**Core**)

Most analysis is not done on the whole table. You often need a subset.

Typical operations:
- select columns
- filter rows
- sort values
- create new variables

In [ ]:
tips[["total_bill", "tip", "day", "time"]].head()

In [ ]:
tips[tips["day"] == "Sun"].head()

In [ ]:
tips.sort_values("total_bill", ascending=False).head(10)

In [ ]:
tips["tip_pct"] = tips["tip"] / tips["total_bill"] * 100
tips.head()

### Interpretation prompt
What is the practical meaning of `tip_pct`?  
Why might it be more informative than raw `tip` amount in some situations?

## 9. Summarizing one categorical variable (**Core**)

For one categorical variable, useful questions are:

- How many observations are in each category?
- What proportion belongs to each category?
- Which category is most common?

In [ ]:
tips["day"].value_counts()

In [ ]:
tips["day"].value_counts(normalize=True).mul(100).round(1)

In [ ]:
tips_missing["sex"].value_counts(dropna=False)

### Plain-language model interpretation
Example:

> In the `tips` dataset, Saturday and Sunday appear more often than Thursday and Friday.  
> If we want to compare restaurant behavior across days, we should remember that the group sizes are not equal.

Now write one sentence of your own for `sex` using `tips_missing`.

## 10. Visualizing one categorical variable (**Core**)

A **bar chart** is the standard graph for one categorical variable.

Why bar chart and not histogram?
- categories are separate groups, not intervals on a number line
- bar charts compare **counts or proportions**
- spaces between bars reflect distinct categories

In [ ]:
day_counts = tips["day"].value_counts().sort_index()

plt.figure(figsize=(7,4))
sns.barplot(x=day_counts.index, y=day_counts.values)
plt.title("Counts by Day")
plt.xlabel("Day")
plt.ylabel("Count")
plt.show()

### Discussion question
Why is a pie chart often less effective than a bar chart for this task?

Possible arguments:
- harder to compare angles than lengths
- too many slices quickly become unreadable
- bar charts make ranking easier to see

## 11. Summarizing one numerical variable (**Core**)

For one numerical variable, we usually care about:

- **center**: mean or median
- **spread**: standard deviation or IQR
- **shape**: symmetric, skewed, unimodal, bimodal
- **outliers**: unusual values

Useful summaries:
- mean
- median
- standard deviation
- quartiles
- IQR
- min / max

In [ ]:
tips["total_bill"].describe()

In [ ]:
summary_total_bill = pd.Series({
    "mean": tips["total_bill"].mean(),
    "median": tips["total_bill"].median(),
    "std": tips["total_bill"].std(),
    "q1": tips["total_bill"].quantile(0.25),
    "q3": tips["total_bill"].quantile(0.75),
    "iqr": tips["total_bill"].quantile(0.75) - tips["total_bill"].quantile(0.25),
    "min": tips["total_bill"].min(),
    "max": tips["total_bill"].max()
})
summary_total_bill

### Practical argument: mean vs median
Suppose most bills are moderate but a few are extremely large. Then:

- the **mean** gets pulled upward by large values
- the **median** is more resistant to extreme values

This is why skewed distributions often call for the **median** and **IQR**.

## 12. How to describe a distribution (**Core**)

A reliable sentence template:

1. **Center**: where is the distribution centered?
2. **Spread**: how variable is it?
3. **Shape**: symmetric or skewed? one peak or several?
4. **Outliers**: any unusual values?

### Example template
> The distribution of `total_bill` is right-skewed, centered around about 18–20 dollars, with moderate spread and a few relatively large values.

## 13. Histograms (**Core**)

A histogram is the standard graph for one **numerical** variable.

It helps us see:
- shape
- skewness
- spread
- possible outliers
- whether multiple groups may be mixed together

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(tips["total_bill"], bins=15, kde=False)
plt.title("Histogram of Total Bill")
plt.xlabel("Total bill")
plt.ylabel("Count")
plt.show()

### Try it yourself
Before reading further, describe this histogram in one sentence using:

- center
- spread
- shape
- outliers

## 14. Density histograms and the area idea (**Recommended**)

When bins have equal width, count histograms are usually fine for basic EDA.

But when you want a scale that represents **distribution shape more fairly**, especially across different bin widths, a **density histogram** is useful.

Key principle:
- In a density histogram, **area** corresponds to proportion of data.
- Total area should be about **1**.

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(tips["total_bill"], bins=15, stat="density", kde=True)
plt.title("Density Histogram of Total Bill")
plt.xlabel("Total bill")
plt.ylabel("Density")
plt.show()

### Why this matters
Bad graphs do not just look messy. They can tell a false story.

If bins have unequal width, comparing bar heights directly can be misleading.  
That is why the **area principle** matters.

## 15. A quick note on skewness and robust summaries (**Core**)

For skewed distributions:
- the mean may be pulled toward the long tail
- the median is often a better summary of a 'typical' case
- the IQR is often more robust than the SD

This is not a rigid rule, but it is a very useful habit.

In [ ]:
tips[["total_bill", "tip", "tip_pct"]].agg(["mean", "median", "std"])

### Discussion question
For which variable in the table above would you trust the **median** more than the **mean**? Why?

## 16. Box plots (**Core**)

A box plot gives a compact summary of a numerical distribution:

- median
- first and third quartiles
- IQR
- potential outliers

It is especially useful for quick comparison across groups.

In [ ]:
plt.figure(figsize=(7,4))
sns.boxplot(x=tips["total_bill"])
plt.title("Box Plot of Total Bill")
plt.xlabel("Total bill")
plt.show()

### Interpretation prompt
How does the box plot support or complement what you saw in the histogram?

Remember:
- histograms show shape more clearly
- box plots summarize center, spread, and outliers more compactly

## 17. Outliers: unusual does not always mean wrong (**Core**)

Outliers can come from several sources:
- genuine rare cases
- measurement or data-entry errors
- mixing of different subpopulations

Good practice:
1. inspect the value
2. ask whether it is plausible
3. decide whether to keep, flag, transform, or investigate further

In [ ]:
q1 = tips["total_bill"].quantile(0.25)
q3 = tips["total_bill"].quantile(0.75)
iqr = q3 - q1
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr

outliers = tips[(tips["total_bill"] < lower_fence) | (tips["total_bill"] > upper_fence)]
outliers[["total_bill", "tip", "day", "time", "size"]].head(10)

### Discussion question
Does being an outlier automatically mean the row is wrong?

Answer: no.  
An outlier is a **statistical description**, not automatic evidence of error.

## 18. Comparing a numerical variable across groups (**Core**)

This is one of the most common EDA tasks.

When you have:
- one **categorical** variable and
- one **numerical** variable

you often want to compare:
- center
- spread
- shape
- outliers

A very good first graph is a **side-by-side box plot**.

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(data=tips, x="day", y="total_bill", order=["Thur", "Fri", "Sat", "Sun"])
plt.title("Total Bill by Day")
plt.xlabel("Day")
plt.ylabel("Total bill")
plt.show()

In [ ]:
tips.groupby("day")["total_bill"].agg(["count", "mean", "median", "std"]).sort_index()

### Plain-language model interpretation
> Bills appear highest on some days, but we should compare both the center and the spread, not just one summary.  
> Group sizes also differ, so a visual and numerical comparison together is safer than relying on one statistic alone.

Now write one sentence of your own.

## 19. Violin plots (**Enrichment**)

A violin plot combines:
- some of the summary benefits of a box plot
- a rough sense of the distribution shape

This is useful, but not required for Week 3 mastery.

In [ ]:
plt.figure(figsize=(8,5))
sns.violinplot(data=tips, x="day", y="total_bill", order=["Thur", "Fri", "Sat", "Sun"])
plt.title("Violin Plot of Total Bill by Day")
plt.xlabel("Day")
plt.ylabel("Total bill")
plt.show()

## 20. Two numerical variables: scatter plots (**Core**)

A scatter plot is the standard graph for two numerical variables.

Each point represents **one row**.

Things to look for:
- direction (positive / negative / none)
- form (linear / curved / clustered)
- strength
- unusual points

In [ ]:
plt.figure(figsize=(7,5))
sns.scatterplot(data=iris, x="sepal_length", y="petal_length", hue="species")
plt.title("Sepal Length vs Petal Length")
plt.xlabel("Sepal length")
plt.ylabel("Petal length")
plt.show()

### Interpretation prompt
What do you notice?

- Is the relationship positive or negative?
- Do species overlap, or do they cluster?
- Would a single average line describe all flowers equally well?

## 21. Scatter plot reading routine (**Core**)

When you see a scatter plot, ask:

1. What does one point represent?
2. What are the two variables?
3. Is the direction positive, negative, or unclear?
4. Does the form look roughly linear?
5. How strong is the pattern?
6. Are there clusters or outliers?

## 22. Pair plots for a quick multivariable overview (**Enrichment**)

A pair plot is a fast way to scan several numerical variables at once.  
It is useful for exploration, but students do not need to master it in Week 3.

In [ ]:
pairplot_cols = ["sepal_length", "sepal_width", "petal_length", "petal_width", "species"]
sns.pairplot(iris[pairplot_cols], hue="species", corner=True)
plt.show()

## 23. Two categorical variables: contingency tables (**Recommended**)

For two categorical variables, a good starting point is a contingency table.

Questions:
- How many cases fall in each combination?
- Are proportions different across groups?

In [ ]:
ct = pd.crosstab(tips["day"], tips["time"])
ct

In [ ]:
pd.crosstab(tips["day"], tips["time"], normalize="index").round(3)

### Interpretation prompt
Why are row proportions often more informative than raw counts here?

## 24. A common practical comparison: tip percentage across groups (**Core**)

Raw tip amounts can increase with bill size.  
A fairer comparison is often **tip percentage**.

This is a good example of why derived variables matter.

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(data=tips, x="day", y="tip_pct", order=["Thur", "Fri", "Sat", "Sun"])
plt.title("Tip Percentage by Day")
plt.xlabel("Day")
plt.ylabel("Tip percentage")
plt.show()

In [ ]:
tips.groupby("day")["tip_pct"].agg(["count", "mean", "median", "std"]).sort_index()

### Discussion question
Why might comparing **tip percentage** be more meaningful than comparing raw `tip` amount?

## 25. Scale matters: a richer example with Gapminder (**Recommended**)

Some variables span a very large range.  
When this happens, raw plots may hide important structure.

`gdpPercap` is a classic example: it is strongly right-skewed.

In [ ]:
gap_2007 = gapminder.query("year == 2007").copy()
gap_2007[["country", "continent", "lifeExp", "pop", "gdpPercap"]].head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(gap_2007["gdpPercap"], bins=30, ax=axes[0])
axes[0].set_title("GDP per capita (raw scale)")
axes[0].set_xlabel("GDP per capita")

sns.histplot(np.log10(gap_2007["gdpPercap"]), bins=30, ax=axes[1])
axes[1].set_title("log10(GDP per capita)")
axes[1].set_xlabel("log10(GDP per capita)")

plt.tight_layout()
plt.show()

### Plain-language lesson
If a variable spans a huge range, always ask whether a transformation such as a log scale might make the structure easier to see.

This is not a trick. It is often necessary for honest interpretation.

## 26. Example of a simple data story (**Recommended**)

Let us compare life expectancy by continent in 2007.

In [ ]:
plt.figure(figsize=(10,5))
sns.boxplot(data=gap_2007, x="continent", y="lifeExp")
plt.title("Life Expectancy by Continent (2007)")
plt.xlabel("Continent")
plt.ylabel("Life expectancy")
plt.xticks(rotation=20)
plt.show()

In [ ]:
gap_2007.groupby("continent")["lifeExp"].agg(["count", "mean", "median", "std"]).round(2)

### Model interpretation
> Life expectancy differs across continents, but the variation within continents is also important.  
> A strong analysis reports both the between-group differences and the within-group spread.

Write your own 2-sentence interpretation before moving on.

### 27. Common mistakes in Week 3 EDA (**Core**)

Watch out for these:

- computing a mean for a categorical variable
- using a histogram for categories
- trusting the mean on strongly skewed data without checking the median
- treating an outlier as an error without evidence
- forgetting that missing values can reduce the sample size
- reporting a graph without any written interpretation
- confusing association with causation

## 28. Reusable EDA template (**Core**)

Use this skeleton whenever you start a new dataset.

In [ ]:
def quick_eda(df):
    print("shape:", df.shape)
    display(df.head())
    display(quality_report(df))
    display(df.describe(include="all").T)

# Example use:
quick_eda(iris)

## 29. Exercises (**Core + Recommended**)

Try to complete these before looking back at earlier cells too much.

### Exercise 1 — variable types
Using `iris`, classify every column as:
- categorical
- numerical
- identifier-like

### Exercise 2 — one numerical variable
Using `tips["tip_pct"]`:
- compute mean, median, SD, Q1, Q3, and IQR
- decide whether mean or median is more trustworthy
- justify your answer in one sentence

### Exercise 3 — one categorical variable
Using `tips["smoker"]`:
- create counts
- create proportions
- make a bar chart
- write one plain-language finding

### Exercise 4 — numerical by categorical
Compare `total_bill` by `time`:
- side-by-side box plot
- grouped summary table
- 2-sentence interpretation

### Exercise 5 — two numerical variables
Using `iris`, create a scatter plot of:
- `petal_length` vs `petal_width`
Color by `species`.
Then describe direction, form, strength, and clustering.

### Exercise 6 — richer interpretation
Using `gap_2007`, compare `gdpPercap` across continents:
- try a box plot
- decide whether raw scale or log scale communicates better
- explain why

## 30. Optional challenge exercises (**Enrichment**)

1. For `tips_missing`, investigate how many rows are lost if you compute summaries on `tip` without handling missing values explicitly.
2. Build a small function that prints:
   - variable type
   - missing count
   - recommended chart type
3. Create one deliberately bad graph and then fix it. Explain what was misleading in the bad version.

## 31. Wrap-up

### What you should now be able to do
- inspect a dataset before analyzing it
- identify cases and variables
- classify variable types
- perform basic data-quality checks
- summarize one variable at a time
- choose a graph that matches variable type
- compare groups using numbers and visuals
- write simple, careful findings from data

### Final reminder
EDA is not a decorative step before 'real analysis.'  
EDA **is** real analysis. It is how you learn what the data can and cannot support.